# Data exploration

Let's understand what's in the data to be able to make an end-to-end compliance-ready pipeline.

In [13]:
import pandas as pd
from pandas.core.frame import DataFrame
from pandas.core.series import Series
from pathlib import Path

In [2]:
DATA_PATH: Path = Path.joinpath(Path.cwd().parent, "data", "raw", "heart_disease_uci.csv")

assert DATA_PATH.exists(), f"Data file not found at {DATA_PATH}"

In [3]:
df: DataFrame = pd.read_csv(DATA_PATH)
print(df.columns, "\n")
print(f"Number of columns: {len(df.columns)}\n")
print(f"Number of entries in the dataframe: {len(df)}")

Index(['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs',
       'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='object') 

Number of columns: 16

Number of entries in the dataframe: 920


# Data Columns

In comparison to the 14 variables that were reported in the [UCI dataset](https://archive.ics.uci.edu/dataset/45/heart+disease), the dataset on [Kaggle](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data?resource=download) has 16.


| Dataset | age | sex | cp  | trestbps | chol | fbs | restecg | thalach | exang | oldpeak | slope | ca  | thal | num | id  | dataset |
| :------ | :-: | :-: | :-: |  :-----: | :--: | :-: | :-----: | :-----: | :---: | :-----: | :---: | :-: | :--: | :-: | :-: | :-----: |
| UCI     |  y  |  y  |  y  |      y   |   y  |  y  |    y    |     y   |   y   |    y    |   y   |  y  |   y  |  y  |  n  |    n    |
| Kaggle  |  y  |  y  |  y  |      y   |   y  |  y  |    y    |     y   |   y   |    y    |   y   |  y  |   y  |  y  |  y  |    y    |


Below I would like to make some assumptions about the data that is present in each of the columns:

- `id (unique id given to each patient):` Each ID must be unique, an integer, and the number of unique IDs must match to the length of the dataframe.
- `age (age of patient in years):` The age of a patient is represented as an integer (int64). Participants must be 18 or older and an upper limit of 120 is set to ensure plausible ages.
- `sex (gender):` Assuming 2 genders (male or female). Should be boolean 0 for male and 1 for female.
- `dataset (location of dataset):` 4 locations from where the dataset is coming from ['Cleveland', 'Hungary', 'Switzerland', 'VA Long Beach'].
- `cp (chest pain type):` 4 types of chest pains ['typical angina', 'asymptomatic', 'non-anginal', 'atypical angina'].
- `trestbps (resting blood pressure):` Resting blood pressures as float, greater equal 80 and less than equal 200.
- `chol (cholesterol measure):` Cholesterol measure as float, greater equal 80 and less than equal 610. Based on extreme observations (85 and 603).
- `fbs (fasting blood sugar):` Is the patient's fasting blood sugar greater than 120 mg/dL? Boolean: True or False.
- `restecg (ECG observation at resting condition):` 3 ECG observations ['lv hypertrophy', 'normal', 'st-t abnormality'].
- `thalch (maximum heart rate achieved):` Maximum heart rate achieved as float. Greater equal 60 and less than equal 220.
- `exang (exercise-induced angine):` Boolean: if physical exertion causes chest pain then True, False otherwise.
- `oldpeak (ST depression induced by exercise relative to rest):` Float: greater equal -3 and less than or equal 10.
- `slope (the slope of the peark exercise ST segment):` 3 possible trends of ST segment: ['downsloping', 'flat', 'upsloping'].
- `ca (number of major vessels colored by fluoroscopy):` 4 possible values (0-3). 0: completely healthy. 1, 2, or 3 signify # of major vessels that are diseased.
- `thal (Thallium stress test results):` 3 possible results ['fixed defect', 'normal', 'reversable defect'].
- `num (5-stage scale of heart disease):` 5 possible values (0-4). Integer. 0 if no disease, 1-4 severity of disease.


In [38]:
def print_info(df: Series) -> None:
    print(df.dtypes)
    print(df.unique())
    print(f"NaN present: {df.isna().any()}")

## id

In [ ]:
id_df: Series = df['id']
print_info(id_df)
assert len(id_df) == len(id_df.unique()), "# of unique ids must match the number of patients"

## age

In [40]:
age_df: Series = df['age']
print_info(age_df)
print(f"All greater equal 18: {(age_df >= 18).all()}\nAll less than or equal 120: {(age_df <= 120).all()}")

int64
[63 67 37 41 56 62 57 53 44 52 48 54 49 64 58 60 50 66 43 40 69 59 42 55
 61 65 71 51 46 45 39 68 47 34 35 29 70 77 38 74 76 28 30 31 32 33 36 72
 73 75]
NaN present: False
All greater equal 18: True
All less than or equal 120: True


## sex

In [41]:
sex_df: Series = df['sex']
sex_df = sex_df.astype('string') # pandas loads it as object type, the dtype should be string but we want to convert it to boolean in the future
print_info(sex_df)

string
<StringArray>
['Male', 'Female']
Length: 2, dtype: string
NaN present: False


## dataset

In [42]:
dataset_df: Series = df['dataset']
dataset_df: Series = dataset_df.astype('string') # pandas loads it as object type, the dtype should be string
print_info(dataset_df)

string
<StringArray>
['Cleveland', 'Hungary', 'Switzerland', 'VA Long Beach']
Length: 4, dtype: string
NaN present: False


## cp

In [43]:
cp_df: Series = df['cp']
cp_df: Series = cp_df.astype('string') # pandas loads it as object type, the dtype should be string
print_info(cp_df)

string
<StringArray>
['typical angina', 'asymptomatic', 'non-anginal', 'atypical angina']
Length: 4, dtype: string
NaN present: False


## trestbps

The value below 80 is 0, an unrealistic value so we will drop this.

In [ ]:
trestbps_df: Series = df['trestbps']

print_info(trestbps_df)
print(f"All greater equal to 80: {(trestbps_df >= 80).all()}\nAll less than equal to 220: {(trestbps_df <= 200).all()}\n")

# remove NaN
trestbps_df: Series = trestbps_df.dropna()
print_info(trestbps_df)
print(f"All greater equal to 80: {(trestbps_df >= 80).all()}\nAll less than equal to 220: {(trestbps_df <= 200).all()}\n")

# # of values below 80
print(f"Number of values below 80: {trestbps_df.where(trestbps_df < 80).count()}")
print(f"Values below 80: {trestbps_df.where(trestbps_df < 80).dropna()}")

float64
[145. 160. 120. 130. 140. 172. 150. 110. 132. 117. 135. 112. 105. 124.
 125. 142. 128. 170. 155. 104. 180. 138. 108. 134. 122. 115. 118. 100.
 200.  94. 165. 102. 152. 101. 126. 174. 148. 178. 158. 192. 129. 144.
 123. 136. 146. 106. 156. 154. 114. 164.  98. 190.  nan 113.  92.  95.
  80. 185. 116.   0.  96. 127.]
NaN present: True
All greater equal to 80: False
All less than equal to 220: False

float64
[145. 160. 120. 130. 140. 172. 150. 110. 132. 117. 135. 112. 105. 124.
 125. 142. 128. 170. 155. 104. 180. 138. 108. 134. 122. 115. 118. 100.
 200.  94. 165. 102. 152. 101. 126. 174. 148. 178. 158. 192. 129. 144.
 123. 136. 146. 106. 156. 154. 114. 164.  98. 190. 113.  92.  95.  80.
 185. 116.   0.  96. 127.]
NaN present: False
All greater equal to 80: False
All less than equal to 220: True

Number of values below 80: 1
Values below 80: 753    0.0
Name: trestbps, dtype: float64


## chol

The values below 100 are 85 and 0. We will swap patients with a chol of 0 to NaN as this was most likely not tracked. We will then impute these values later. We will drop the patient with value of 85.
The value above 600 is 603. We will drop this patient.


In [63]:
chol_df: Series = df['chol']

print_info(chol_df)
print(f"All greater equal to 100: {(chol_df >= 100).all()}\nAll less than equal to 600: {(chol_df <= 600).all()}\n")

# remove NaN
chol_df: Series = chol_df.dropna()
print_info(chol_df)
print(f"All greater equal to 100: {(chol_df >= 100).all()}\nAll less than equal to 600: {(chol_df <= 600).all()}\n")

# # of values below 100
print(chol_df.where(chol_df < 100).count())
print(chol_df.where(chol_df < 100).dropna().unique(), "\n")

print(chol_df.where(chol_df > 600).count())
print(chol_df.where(chol_df > 600).dropna())

float64
[233. 286. 229. 250. 204. 236. 268. 354. 254. 203. 192. 294. 256. 263.
 199. 168. 239. 275. 266. 211. 283. 284. 224. 206. 219. 340. 226. 247.
 167. 230. 335. 234. 177. 276. 353. 243. 225. 302. 212. 330. 175. 417.
 197. 198. 290. 253. 172. 273. 213. 305. 216. 304. 188. 282. 185. 232.
 326. 231. 269. 267. 248. 360. 258. 308. 245. 270. 208. 264. 321. 274.
 325. 235. 257. 164. 141. 252. 255. 201. 222. 260. 182. 303. 265. 309.
 307. 249. 186. 341. 183. 407. 217. 288. 220. 209. 227. 261. 174. 281.
 221. 205. 240. 289. 318. 298. 564. 246. 322. 299. 300. 293. 277. 214.
 207. 223. 160. 394. 184. 315. 409. 244. 195. 196. 126. 313. 259. 200.
 262. 215. 228. 193. 271. 210. 327. 149. 295. 306. 178. 237. 218. 242.
 319. 166. 180. 311. 278. 342. 169. 187. 157. 176. 241. 131. 132.  nan
 161. 173. 194. 297. 292. 339. 147. 291. 358. 412. 238. 163. 280. 202.
 328. 129. 190. 179. 272. 100. 468. 320. 312. 171. 365. 344.  85. 347.
 251. 287. 156. 117. 466. 338. 529. 392. 329. 355. 603. 404. 518. 285

## fbs


In [68]:
fbs_df: Series = df['fbs'] # should be bool in the future

print_info(fbs_df)

print('\n')
fbs_df: Series = fbs_df.dropna()
fbs_df: Series = fbs_df.astype('bool')
print_info (fbs_df)

object
[True False nan]
NaN present: True


bool
[ True False]
NaN present: False


## restecg


In [72]:
restecg_df: Series = df['restecg']
restecg_df: Series = restecg_df.astype('string')

print_info(restecg_df)

print('\n')

restecg_df: Series = restecg_df.dropna()
print_info(restecg_df)


string
<StringArray>
['lv hypertrophy', 'normal', 'st-t abnormality', <NA>]
Length: 4, dtype: string
NaN present: True


string
<StringArray>
['lv hypertrophy', 'normal', 'st-t abnormality']
Length: 3, dtype: string
NaN present: False


## thalch

In [75]:
thalch_df: Series = df['thalch']

print_info(thalch_df)
print(f"All greater equal to 60: {(thalch_df >= 60).all()}\nAll less than equal to 220: {(thalch_df <= 220).all()}\n")

thalch_df: Series = thalch_df.dropna()
print_info(thalch_df)
print(f"All greater equal to 60: {(thalch_df >= 60).all()}\nAll less than equal to 220: {(thalch_df <= 220).all()}\n")

float64
[150. 108. 129. 187. 172. 178. 160. 163. 147. 155. 148. 153. 142. 173.
 162. 174. 168. 139. 171. 144. 132. 158. 114. 151. 161. 179. 120. 112.
 137. 157. 169. 165. 123. 128. 152. 140. 188. 109. 125. 131. 170. 113.
  99. 177. 141. 180. 111. 143. 182. 156. 115. 149. 145. 146. 175. 186.
 185. 159. 130. 190. 136.  97. 127. 154. 133. 126. 202. 103. 166. 164.
 184. 124. 122.  96. 138.  88. 105. 194. 195. 106. 167.  95. 192. 117.
 121. 116.  71. 118. 181. 134.  90.  98. 176. 135. 110.  nan 100.  87.
 102.  92.  91.  82. 119.  94. 104.  60.  83.  63.  70.  77.  72.  78.
  86.  93.  67.  84.  80. 107.  69.  73.]
NaN present: True
All greater equal to 60: False
All less than equal to 220: False

float64
[150. 108. 129. 187. 172. 178. 160. 163. 147. 155. 148. 153. 142. 173.
 162. 174. 168. 139. 171. 144. 132. 158. 114. 151. 161. 179. 120. 112.
 137. 157. 169. 165. 123. 128. 152. 140. 188. 109. 125. 131. 170. 113.
  99. 177. 141. 180. 111. 143. 182. 156. 115. 149. 145. 146. 175. 186.
 185. 

## exang


In [78]:
exang_df: Series = df['exang']

print_info(exang_df)

print('\n')

exang_df: Series = exang_df.dropna()
exang_df: Series = exang_df.astype('bool')
print_info(exang_df)

object
[False True nan]
NaN present: True


bool
[False  True]
NaN present: False


## oldpeak

In [81]:
oldpeak_df: Series = df['oldpeak']

print_info(oldpeak_df)
print(f"All greater equal to -3: {(oldpeak_df >= -3).all()}\nAll less than equal to 10: {(oldpeak_df <= 10).all()}\n")

oldpeak_df: Series = oldpeak_df.dropna()
print(f"All greater equal to -3: {(oldpeak_df >= -3).all()}\nAll less than equal to 10: {(oldpeak_df <= 10).all()}\n")

float64
[ 2.3  1.5  2.6  3.5  1.4  0.8  3.6  0.6  3.1  0.4  1.3  0.   0.5  1.6
  1.   1.2  0.2  1.8  3.2  2.4  2.   2.5  2.2  2.8  3.   3.4  6.2  4.
  5.6  2.9  0.1  2.1  1.9  4.2  0.9  1.1  3.8  0.7  0.3  4.4  5.   nan
 -1.1 -1.5 -0.1 -2.6 -0.7 -2.  -1.   1.7 -0.8 -0.5 -0.9  3.7]
NaN present: True
All greater equal to -3: False
All less than equal to 10: False

All greater equal to -3: True
All less than equal to 10: True



## slope


In [83]:
slope_df: Series = df['slope']

print_info(slope_df)

print("\n")

slope_df: Series = slope_df.dropna()
print_info(slope_df)

object
['downsloping' 'flat' 'upsloping' nan]
NaN present: True


object
['downsloping' 'flat' 'upsloping']
NaN present: False


## ca


In [89]:
ca_df: Series = df['ca']

print_info(ca_df)

print('\n')

ca_df: Series = ca_df.dropna()
print_info(ca_df)

assert ca_df.isin([0., 1., 2., 3.]).all()


float64
[ 0.  3.  2.  1. nan]
NaN present: True


float64
[0. 3. 2. 1.]
NaN present: False


## thal


In [93]:
thal_df: Series = df['thal']
thal_df: Series = thal_df.astype('string')

print_info(thal_df)

thal_df: Series = thal_df.dropna()
print_info(thal_df)
assert thal_df.isin(['fixed defect', 'normal', 'reversable defect']).all()

string
<StringArray>
['fixed defect', 'normal', 'reversable defect', <NA>]
Length: 4, dtype: string
NaN present: True
string
<StringArray>
['fixed defect', 'normal', 'reversable defect']
Length: 3, dtype: string
NaN present: False


## num

In [95]:
num_df: Series = df['num']

print_info(num_df)
assert num_df.isin([0, 1, 2, 3, 4]).all()

int64
[0 2 1 3 4]
NaN present: False
